In [5]:
# Complete script with explicit RandomizedSearchCV integration for MLP 
# (inserted immediately after your existing imports and split)

import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, matthews_corrcoef, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# 1) Load & balance
df = pd.read_csv('ai4i2020.csv')
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
X = df[feature_cols]
y = df["Machine failure"]

# try imblearn, fallback to pandas sampling
try:
    from imblearn.under_sampling import RandomUnderSampler
    rus = RandomUnderSampler(sampling_strategy={0:339, 1:339}, random_state=42)
    X_bal, y_bal = rus.fit_resample(X, y)
except ImportError:
    df_min = df[df['Machine failure']==1].sample(339, random_state=42)
    df_maj = df[df['Machine failure']==0].sample(339, random_state=42)
    df_bal = pd.concat([df_min, df_maj]).sample(frac=1, random_state=42)
    X_bal, y_bal = df_bal[feature_cols], df_bal["Machine failure"]

# 2) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

# 3) Preprocessing
preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), feature_cols)
])
mcc_scorer = make_scorer(matthews_corrcoef)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Example: RandomizedSearchCV for MLPClassifier with early stopping
#    Insert this block right after your split & preprocessing setup.

# Define MLP pipeline with early stopping enabled
mlp_pipeline = Pipeline([
    ("scale", preprocessor),
    ("clf", MLPClassifier(
       # max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# Parameter distributions for RandomizedSearch
mlp_param_dist = {
    "clf__hidden_layer_sizes": [(50,), (100,), (50,50), (100,50)],
    "clf__activation": ["relu", "tanh", "logistic"],
    "clf__learning_rate": ["constant", "adaptive"],
    "clf__alpha": [1e-5, 1e-4, 1e-3],
    "clf__n_iter_no_change": [5, 10, 20],
    "clf__validation_fraction": [0.1, 0.2],
    "clf__max_iter": [200, 500, 1000] 
}

# Instantiate RandomizedSearchCV
mlp_random_search = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=mlp_param_dist,
    n_iter=500,
    scoring=mcc_scorer,
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Run the search
print("\n--- Running RandomizedSearchCV for MLPClassifier ---")
mlp_random_search.fit(X_train, y_train)
print("\nMLP RandomizedSearchCV best CV MCC: "
      f"{mlp_random_search.best_score_:.4f}")
print("MLP best params:", mlp_random_search.best_params_)

# ─────────────────────────────────────────────────────────────────────────────
# 5) (Optional) Continue with GridSearch or RandomizedSearch for the other models
#    as in your existing loop over search_configs...

# 6) Finally, evaluate your tuned MLP on the test set
y_pred_mlp = mlp_random_search.predict(X_test)
print("\nMLP Test MCC:", matthews_corrcoef(y_test, y_pred_mlp))
print(classification_report(y_test, y_pred_mlp))



--- Running RandomizedSearchCV for MLPClassifier ---
Fitting 5 folds for each of 500 candidates, totalling 2500 fits

MLP RandomizedSearchCV best CV MCC: 0.6581
MLP best params: {'clf__validation_fraction': 0.2, 'clf__n_iter_no_change': 20, 'clf__max_iter': 1000, 'clf__learning_rate': 'constant', 'clf__hidden_layer_sizes': (100, 50), 'clf__alpha': 0.001, 'clf__activation': 'tanh'}

MLP Test MCC: 0.6918500955502519
              precision    recall  f1-score   support

           0       0.83      0.87      0.85        68
           1       0.86      0.82      0.84        68

    accuracy                           0.85       136
   macro avg       0.85      0.85      0.85       136
weighted avg       0.85      0.85      0.85       136

